<a href="https://colab.research.google.com/github/iZevro/ITCS-3162/blob/Lab-3/lab_03_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Example: Data Wrangling with the Diamonds Dataset

**ITCS 3162 — Introduction to Data Mining**
**Companion to Zybooks Chapter 3 (Data Wrangling)**

Real-world data is rarely clean. "Wrangling" is the unglamorous-but-essential work of getting raw data into a shape you can actually analyze — fixing types, handling missing values, removing duplicates, and reshaping rows and columns.

This walkthrough uses the **Diamonds dataset** (the same one your Zybooks chapter uses): roughly 54,000 diamonds with their carat, cut, color, clarity, dimensions, and price.

## Learning objectives
By the end you will be able to:
1. Load a dataset with `pandas` from a built-in source or a CSV
2. Inspect data types, shape, and basic structure with `info()`, `head()`, `shape`, `dtypes`
3. Find and handle missing values
4. Detect and remove duplicates
5. Filter rows and select columns
6. Create derived columns (feature engineering preview)
7. Group and summarize with `groupby()`


## 1. Load the data

The diamonds dataset ships with seaborn — no download needed. Run the cell below.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("diamonds")

# Ensure ordered category dtypes for the three quality columns. seaborn already
# does this, but doing it explicitly makes the code portable to a plain CSV load
# and teaches you about ordered categoricals.
df["cut"]     = pd.Categorical(df["cut"],
                  categories=["Fair","Good","Very Good","Premium","Ideal"], ordered=True)
df["color"]   = pd.Categorical(df["color"],
                  categories=list("JIHGFED"), ordered=True)  # J=worst, D=best
df["clarity"] = pd.Categorical(df["clarity"],
                  categories=["I1","SI2","SI1","VS2","VS1","VVS2","VVS1","IF"], ordered=True)

df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


### Column dictionary

| Column | Type | Meaning |
|---|---|---|
| `carat` | float | Weight of the diamond |
| `cut`   | categorical | Quality of the cut (Fair < Good < Very Good < Premium < Ideal) |
| `color` | categorical | D (best) through J (worst) |
| `clarity` | categorical | I1 (worst) through IF (best) |
| `depth` | float | Total depth percentage |
| `table` | float | Width of top relative to widest point |
| `price` | int | Price in USD |
| `x`, `y`, `z` | float | Length, width, depth in mm |


## 2. First-look inspection

Three commands you'll run on *every* new dataset:


In [2]:
print("Shape (rows, cols):", df.shape)
print("Memory used:", df.memory_usage(deep=True).sum() / 1024**2, "MB")

Shape (rows, cols): (53940, 10)
Memory used: 3.036884307861328 MB


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   price    53940 non-null  int64   
 7   x        53940 non-null  float64 
 8   y        53940 non-null  float64 
 9   z        53940 non-null  float64 
dtypes: category(3), float64(6), int64(1)
memory usage: 3.0 MB


`info()` tells you the column types and how many non-null entries each has. `category` is a memory-efficient dtype for repeated strings — it's like an enum.


In [4]:
df.describe()

,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


`describe()` summarizes numeric columns. Notice the strange minimums for `x`, `y`, `z` — diamonds with zero dimensions are physically impossible, hinting at data quality issues we'll investigate.

To summarize the categorical columns separately:


In [5]:
df.describe(include="category")

,cut,color,clarity
count,53940,53940,53940
unique,5,7,8
top,Ideal,G,SI1
freq,21551,11292,13065


## 3. Missing values

In this dataset there aren't any nulls, but checking is the habit:


In [6]:
df.isna().sum()

,0
carat,0
cut,0
color,0
clarity,0
depth,0
table,0
price,0
x,0
y,0
z,0


When you *do* find missing values, the three usual options are:
- **Drop** rows: `df.dropna()`
- **Drop** columns that are mostly missing: `df.drop(columns=["bad_col"])`
- **Fill** with a sensible value: `df["age"].fillna(df["age"].median())`

The right choice depends on *why* the values are missing (you'll see this in Lab 5).


## 4. Duplicates

Are there any exact duplicates?


In [7]:
dups = df.duplicated().sum()
print(f"Duplicate rows: {dups}")

# Remove them (returns a new DataFrame; original is unchanged)
df_dedup = df.drop_duplicates()
print(f"After dedup: {df_dedup.shape}")

Duplicate rows: 146
After dedup: (53794, 10)


## 5. Data quality: impossible values

Earlier `describe()` showed `x`, `y`, `z` minimums of zero. Let's find those rows.


In [8]:
bad = df[(df["x"] == 0) | (df["y"] == 0) | (df["z"] == 0)]
print(f"Rows with a zero dimension: {len(bad)}")
bad.head()

Rows with a zero dimension: 20


,carat,cut,color,clarity,depth,table,price,x,y,z
2207,1.00,Premium,G,SI2,59.1,59.0,3142,6.55,6.48,0.0
2314,1.01,Premium,H,I1,58.1,59.0,3167,6.66,6.60,0.0
4791,1.10,Premium,G,SI2,63.0,59.0,3696,6.50,6.47,0.0
5471,1.01,Premium,F,SI2,59.2,58.0,3837,6.50,6.47,0.0
10167,1.50,Good,G,I1,64.0,61.0,4731,7.15,7.04,0.0


These are almost certainly measurement errors. A reasonable wrangling step is to drop them.


In [9]:
df_clean = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)].copy()
print(f"Before: {len(df)} rows | After: {len(df_clean)} rows | Removed: {len(df) - len(df_clean)}")

Before: 53940 rows | After: 53920 rows | Removed: 20


> **Why `.copy()`?** Without it, `df_clean` would be a *view* into `df`, and modifying it later would trigger pandas' `SettingWithCopyWarning`. Use `.copy()` whenever you take a subset you intend to modify.


## 6. Filtering and selecting

Pull out only the columns you care about, only the rows that matter.


In [10]:
# Just two columns
df_clean[["carat", "price"]].head()

,carat,price
0,0.23,326
1,0.21,326
2,0.23,327
3,0.29,334
4,0.31,335


In [11]:
# Boolean filtering — "Ideal cut, larger than 1 carat"
mask = (df_clean["cut"] == "Ideal") & (df_clean["carat"] > 1.0)
ideal_big = df_clean[mask]
print(f"Ideal-cut diamonds over 1 carat: {len(ideal_big)}")
ideal_big.head()

Ideal-cut diamonds over 1 carat: 5659


,carat,cut,color,clarity,depth,table,price,x,y,z
653,1.01,Ideal,I,I1,61.5,57.0,2844,6.45,6.46,3.97
715,1.02,Ideal,H,SI2,61.6,55.0,2856,6.49,6.43,3.98
865,1.02,Ideal,I,I1,61.7,56.0,2872,6.44,6.49,3.99
918,1.02,Ideal,J,SI2,60.3,54.0,2879,6.53,6.50,3.93
992,1.01,Ideal,I,I1,61.5,57.0,2896,6.46,6.45,3.97


## 7. Creating new columns (feature engineering preview)

A derived feature can be more useful than the raw inputs. Let's add **price per carat** (a more comparable measure of value) and a **volume** column.


In [12]:
df_clean["price_per_carat"] = df_clean["price"] / df_clean["carat"]
df_clean["volume_mm3"] = df_clean["x"] * df_clean["y"] * df_clean["z"]

df_clean[["carat", "price", "price_per_carat", "volume_mm3"]].head()

,carat,price,price_per_carat,volume_mm3
0,0.23,326,1417.391304,38.202030
1,0.21,326,1552.380952,34.505856
2,0.23,327,1421.739130,38.076885
3,0.29,334,1151.724138,46.724580
4,0.31,335,1080.645161,51.917250


## 8. Grouping and summarizing

The single most useful pandas operation: **split** by a category, **apply** an aggregation, **combine** into a summary.


In [13]:
# Average price per cut category
df_clean.groupby("cut", observed=True)["price"].mean().sort_values(ascending=False)

,price
cut,
Premium,4579.684543
Fair,4357.500932
Very Good,3981.664101
Good,3926.403509
Ideal,3456.941201


Surprising! "Ideal" cuts have the *lowest* average price. That's not because they're worth less — it's because most Ideal-cut diamonds are smaller (carat affects price far more than cut). This is the kind of insight wrangling enables but a chart will make obvious in Lab 4.


In [14]:
# Multiple aggregations at once
df_clean.groupby("cut", observed=True).agg(
    n=("price", "size"),
    avg_carat=("carat", "mean"),
    avg_price=("price", "mean"),
    median_price=("price", "median"),
).round(2)

,n,avg_carat,avg_price,median_price
cut,,,,
Fair,1609,1.05,4357.50,3282.0
Good,4902,0.85,3926.40,3050.5
Very Good,12081,0.81,3981.66,2647.0
Premium,13780,0.89,4579.68,3182.0
Ideal,21548,0.70,3456.94,1809.5


Two categories at once — cross-tabulation:


In [15]:
(df_clean.groupby(["cut", "color"], observed=True)["price"]
        .mean()
        .unstack()
        .round(0))

color,J,I,H,G,F,E,D
cut,,,,,,,
Fair,4976.0,4685.0,5136.0,4232.0,3827.0,3682.0,4291.0
Good,4574.0,5079.0,4276.0,4106.0,3499.0,3424.0,3405.0
Very Good,5104.0,5256.0,4535.0,3873.0,3779.0,3215.0,3470.0
Premium,6295.0,5940.0,5198.0,4502.0,4325.0,3539.0,3624.0
Ideal,4918.0,4452.0,3889.0,3718.0,3375.0,2598.0,2629.0


## 9. A quick visual sanity check

We'll go much deeper on visualization in Lab 4 — but a single chart often catches issues a summary table hides.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_clean["carat"], df_clean["price"], alpha=0.1, s=5)
ax.set_xlabel("Carat")
ax.set_ylabel("Price (USD)")
ax.set_title("Diamond price vs. carat (after wrangling)")
plt.show()

The relationship is clearly non-linear and there's heteroscedasticity (variance grows with carat). Wrangling produced a dataset we can now actually model.


## Summary

The wrangling workflow you just walked through:

1. **Load** → `pd.read_csv` / `sns.load_dataset`
2. **Inspect** → `shape`, `info()`, `describe()`, `head()`
3. **Check missing** → `isna().sum()`
4. **Check duplicates** → `duplicated().sum()`, `drop_duplicates()`
5. **Check impossible values** → boolean filters
6. **Filter and select** → `df[mask]`, `df[cols]`
7. **Derive new columns** → arithmetic on existing columns
8. **Summarize** → `groupby().agg()`

Every dataset for the rest of the semester will start with some subset of these steps.

Next: open **`lab_03_exercise.ipynb`** and apply this workflow to the Google Play Store dataset.
